In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
#modelling
from sklearn.metrics import mean_squared_error,r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.svm import SVR 
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

In [5]:
#import csv data as pandas dataframe 
df=pd.read_csv("datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [6]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [7]:
x=df.drop('PaperlessBilling',axis=1)

In [8]:
x.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Electronic check,70.70,151.65,Yes


In [9]:
y=df['PaperlessBilling']

In [10]:
y.head()

0    Yes
1     No
2    Yes
3     No
4    Yes
Name: PaperlessBilling, dtype: object

In [11]:
#create column transformer with 3 types of transformer 
num_features=x.select_dtypes(exclude="object").columns
cat_features=x.select_dtypes(include="object").columns

from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer=StandardScaler()
oh_transformer=OneHotEncoder()

preprocessor =ColumnTransformer(
    [
        ("OneHotEncoder",oh_transformer,cat_features),
        ("StandardScaler",numeric_transformer,num_features),
    ]
)

In [12]:
print(cat_features)

Index(['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaymentMethod', 'TotalCharges', 'Churn'],
      dtype='object')


In [13]:
print(x.dtypes)

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object


In [14]:
x=preprocessor.fit_transform(x)

In [15]:
x.shape

(7043, 13618)

In [16]:
#separate dataset into train and test 
from sklearn.preprocessing import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
x_train.shape,x_test.shape

ImportError: cannot import name 'train_test_split' from 'sklearn.preprocessing' (c:\Program Files\Python312\Lib\site-packages\sklearn\preprocessing\__init__.py)

In [17]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
print(x_train.shape)
print(x_test.shape)

(5634, 13618)
(1409, 13618)


In [ ]:
models={
    "Linear Regressor": LinearRegression(),
    "Lasso":Lasso(),
    "Ridge":Ridge(),
    "K-Neighbors Regressor":KNeighborsRegressor(),
    "Decision Tree":DecisionTreeRegressor(),
    "Random Forest Regressor":RandomForestRegressor(),
    "XGBRegressor":XGBRegressor(),
    "CatBoosting Regressor":CatBoostRegressor(verbose=False),
    "AdaBoost Regressor":AdaBoostRegressor()


}
model_list=[]
r2_list=[]

for i in range(len(list(models))):
    model=list(models.value())[i]
    model.fit(x_train,y_train)#train model

    #make prediction 

y_train_pred=model.predict(x_train)
y_test_pred=model.predict(x_test)

#evaluate Train and Test 
model_train_mae,model_train_rmse,model_train_r2=evaluate_model(y_train,y_train_pred)
model_test_mae,model_test_rmse,model_test_r2=evaluate_model(y_train,y_train_pred)
model_test_mae
print(list(models.keys())[i])
model_list.append(list(models.keys())[i])
print('Model performance for training set')
print("_Root  Mean squared error:{:.4f}.format(model(trainer_rmse))")
print("_root absolute error:{:.4f}".format(model_test_mae))
print("_R2 Score:{:4f}".format(model_train_mae))
print("-R2 score:{:4f}".format(model_train_r2))
print('----------------------')
print('model performance for test set')
print("-Root Mean squared error :{:4f}".format(model_test_rmse))
print("-mean absolute error:{:.4f}".format(model_test_mae))
print('-R2 score :{:4f}'.format(model_test_r2))
r2_list.append(model_test_r2)
print("="*35)
print("\n")

#RESULT 
pd.DataFrame(list(zip(model_list,r2_list))),columns=['Model Name ','R2_score']).sort_values(by=["R2_score"],ascending=False)

#linear Regression 

lin_model=LinearRegression(fit_intercept=True)
lin_model=lin_model.fit(x_train,y_train)
y_pred=lin_model.pred(x_test)
score=r2_score(y_test,y_pred)*100
print("Accuracy of the model is %2f"%score)
#plot y_pred and y_test
plt.scatter(y_test,y_pred);
plt.xlabel('actual')
plt.ylabel('Predicted')
sns.regplot(x=y_test,y=y_pred,ci=None,color='red');

#dif between actual and predicted value 
pred_df=pd.DataFrame({'actual value':y_test,'Predicted value':y_pred})
pred_df

